# Initial Exploration

An initial exploration, for which the Jupyter notebook is better suited, is crucial to understand the structure and content of the datasets we are working with. This step allows us to identify the variables available, their types, and the relationships between them. It also helps us to detect any potential issues with the data, such as missing values or inconsistencies, which could affect our analysis.
The pipeline is particularly complex since it involves dealing with multiple datasets, each with its own structure and coding system and different variables, all of which have different codes which can be difficult to interpret, it can be hard to understand what they measure, if they can be used and compared together and if they are raw measures or simply indexes. Therefore, it is crucial to have a clear understanding of the datasets and the variables they contain before proceeding with the analysis. This involves exploring the datasets, understanding the meaning of the variables, and determining how they can be used in the context of our analysis.Also, the project involves analyzing growth accounts and intangibles data, which are categorized by NACE codes and IO tables, which instead use ICIO codes. Hence, a crucial step in our analysis is to map the NACE codes from the growth accounts to the ICIO codes used in the IO tables. This mapping process is complex and requires careful handling to ensure accurate analysis.

In [14]:
import pandas as pd
import os

# Set up paths
data_raw_path = os.path.join('..', 'data', 'raw')
print(f"Data raw path:{data_raw_path}")

Data raw path:..\data\raw


## 1) Sector Codes Analysis

Load the growth account CSV file and the intangibles analytical files into a pandas DataFrame and display basic information about the dataset structure. In order to understand it, we visualize the first few rows of the dataset. Then, we print the unique sector codes.

In [18]:
# Load growth accounts CSV
growth_accounts_path = os.path.join(data_raw_path, 'growth accounts.csv')
df_growth = pd.read_csv(growth_accounts_path)

# Load intangibles analytical file
intangibles_path = os.path.join(data_raw_path, 'intangibles analytical.csv')
df_intangibles = pd.read_csv(intangibles_path)

print("Growth Accounts Dataset Shape:", df_growth.shape)
print("Intangibles Dataset Shape:", df_intangibles.shape)


def print_code_list(title, codes, per_line=12):
    codes = sorted(pd.Series(codes).dropna().astype(str).unique().tolist())
    print(f"\n{title} ({len(codes)} codes):")
    for i in range(0, len(codes), per_line):
        print("  " + ", ".join(codes[i:i + per_line]))


# NACE sector codes in the two EUKLEMS datasets
unique_nace = df_growth['nace_r2_code'].unique()
unique_nace_intangibles = df_intangibles['nace_r2_code'].unique()

print_code_list("Growth Accounts NACE sectors", unique_nace)
print_code_list("Intangibles Analytical NACE sectors", unique_nace_intangibles)

common_nace = set(unique_nace).intersection(set(unique_nace_intangibles))
print(f"\nCommon NACE codes across both datasets: {len(common_nace)}")
if set(unique_nace) == set(unique_nace_intangibles):
    print("The NACE codes in both datasets match exactly.")
else:
    only_growth = sorted(set(unique_nace) - set(unique_nace_intangibles))
    only_intangibles = sorted(set(unique_nace_intangibles) - set(unique_nace))
    print_code_list("Only in Growth Accounts", only_growth)
    print_code_list("Only in Intangibles Analytical", only_intangibles)

Growth Accounts Dataset Shape: (2467044, 8)
Intangibles Dataset Shape: (61074, 134)

Growth Accounts NACE sectors (58 codes):
  A, B, C, C10-C12, C13-C15, C16-C18, C19, C20, C20-C21, C21, C22-C23, C24-C25
  C26, C26-C27, C27, C28, C29-C30, C31-C33, D, D-E, E, F, G, G45
  G46, G47, H, H49, H50, H51, H52, H53, I, J, J58-J60, J61
  J62-J63, K, L, L68A, M, M-N, MARKT, MARKTxAG, N, O, O-Q, P
  Q, Q86, Q87-Q88, R, R-S, S, T, TOT, TOT_IND, U

Intangibles Analytical NACE sectors (58 codes):
  A, B, C, C10-C12, C13-C15, C16-C18, C19, C20, C20-C21, C21, C22-C23, C24-C25
  C26, C26-C27, C27, C28, C29-C30, C31-C33, D, D-E, E, F, G, G45
  G46, G47, H, H49, H50, H51, H52, H53, I, J, J58-J60, J61
  J62-J63, K, L, L68A, M, M-N, MARKT, MARKTxAG, N, O, O-Q, P
  Q, Q86, Q87-Q88, R, R-S, S, T, TOT, TOT_IND, U

Common NACE codes across both datasets: 58
The NACE codes in both datasets match exactly.


Load the intangibles analytical CSV file and extract unique sectors from it.

In [19]:
# Load ICIO sector codes used for input-output network analysis
from utils.constants import ICIO_SECTOR_CODES

unique_icio = sorted(ICIO_SECTOR_CODES)
print_code_list("ICIO sectors", unique_icio)


ICIO sectors (50 codes):
  A01, A02, A03, B05, B06, B07, B08, B09, C10T12, C13T15, C16, C17_18
  C19, C20, C21, C22, C23, C24A, C24B, C25, C26, C27, C28, C29
  C301, C302T309, C31T33, D, E, F, G, H49, H50, H51, H52, H53
  I, J58T60, J61, J62_63, K, L, M, N, O, P, Q, R
  S, T


The first two datasets follow the same NACE convention, while the IO tables follow the ICIO convention. 

## 2) Variables Analysis (Italy Only)

This section gives a clear overview of the variables available in the EUKLEMS datasets for Italy (geo_code equals IT).

It shows the unique variables in Growth Accounts and Intangibles Analytical.
It also shows yearly summary statistics for each variable: missing values, mean, median, minimum, and maximum.

In [21]:
df_growth_it = df_growth[df_growth["geo_code"] == "IT"].copy()
df_intangibles_it = df_intangibles[df_intangibles["geo_code"] == "IT"].copy()

for _df in (df_growth_it, df_intangibles_it):
    _df["year"] = pd.to_numeric(_df["year"], errors="coerce")

# Select only years 2016-2021
df_growth_it = df_growth_it[df_growth_it["year"].between(2016, 2021)]
df_intangibles_it = df_intangibles_it[df_intangibles_it["year"].between(2016, 2021)]

print("Italy rows in growth accounts:", len(df_growth_it))
print("Italy rows in intangibles analytical:", len(df_intangibles_it))

growth_vars = sorted(df_growth_it["var"].dropna().astype(str).unique()) if "var" in df_growth_it.columns else []
id_cols = {"geo_code", "geo_name", "nace_r2_code", "nace_r2_name", "year"}
intangibles_vars = sorted(c for c in df_intangibles_it.columns if c not in id_cols)
all_vars = sorted(set(growth_vars).union(intangibles_vars))

print("\nGrowth Accounts unique variables (Italy):", len(growth_vars))
print(growth_vars)
print("\nIntangibles Analytical unique variables (Italy):", len(intangibles_vars))
print(intangibles_vars)
print("\nCombined unique variables across both datasets (Italy):", len(all_vars))
print(all_vars)

TARGET = ("Soft_DB", "VA_CP")                          
                                                                                                                                                                                                                     
df_growth_it = df_growth_it[df_growth_it["var"].str.contains("|".join(TARGET), na=False)]
print("\nGrowth Accounts (Italy) — matching variables:")                                                                                                                                                           
print(df_growth_it["var"].unique())
                                                                                                                                                                                                                     
id_cols = ["year", "nace_r2_code", "geo_code", "geo_name", "nace_r2_name"]
id_cols = [c for c in id_cols if c in df_intangibles_it.columns]                                                                                                                                                 
data_cols = [c for c in df_intangibles_it.columns if any(t in c for t in TARGET)]                                                                                                                                  
                                                                                                                                                                                                                     
df_intangibles_it = df_intangibles_it[id_cols + data_cols]                                                                                                                                                                                                      
print("\nIntangibles Analytical (Italy) — matching columns:")                                                                                                                                                      
print(data_cols)

Italy rows in growth accounts: 16272
Italy rows in intangibles analytical: 348

Growth Accounts unique variables (Italy): 68
['CAP', 'CAPEconComp_QI', 'CAPICT_QI', 'CAPInnovprop_QI', 'CAPIntang_QI', 'CAPNICT_QI', 'CAPTang_QI', 'CAP_QI', 'LAB', 'LAB_QI', 'LP1ConEconComp', 'LP1ConInnovprop', 'LP1ConIntang', 'LP1ConIntangNA', 'LP1ConIntang_BU', 'LP1ConIntangnonNA', 'LP1ConLC', 'LP1ConLC_BU', 'LP1ConTFP', 'LP1ConTFP_BU', 'LP1ConTangICT', 'LP1ConTangICT_BU', 'LP1ConTangNICT', 'LP1ConTangNICT_BU', 'LP1Con_Soft_DB', 'LP1TFP_BU_I', 'LP1TFP_I', 'LP1_BU_G', 'LP1_G', 'LP2ConEconComp', 'LP2ConIntang', 'LP2ConIntangNA', 'LP2ConIntang_BU', 'LP2ConIntangnonNA', 'LP2ConLC', 'LP2ConLC_BU', 'LP2ConTFP', 'LP2ConTFP_BU', 'LP2ConTangICT', 'LP2ConTangICT_BU', 'LP2ConTangNICT', 'LP2ConTangNICT_BU', 'LP2Con_Soft_DB', 'LP2TFP_BU_I', 'LP2TFP_I', 'LP2_BU_G', 'LP2_G', 'VAConEconComp', 'VAConH', 'VAConH_BU', 'VAConInnovprop', 'VAConIntang', 'VAConIntangNA', 'VAConIntang_BU', 'VAConIntangnonNA', 'VAConLC', 'VAConLC

In [ ]:
# ── Growth accounts stats (long format) ───────────────────────────────────
print("\n" + "="*60)
print("GROWTH ACCOUNTS — yearly stats by variable")
print("="*60)

growth_stats = (
    df_growth_it.groupby(["var", "year"])["value"]
    .agg(
        n="count",
        missing=lambda s: s.isna().sum(),
        mean="mean",
        median="median",
        std="std",
        min="min",
        max="max",
    )
    .reset_index()
    .sort_values(["var", "year"])
)

for var, grp in growth_stats.groupby("var"):
    print(f"\n— {var} —")
    print(grp.drop(columns="var").to_string(index=False))

# ── Intangibles stats (wide format) ───────────────────────────────────────
print("\n" + "="*60)
print("INTANGIBLES ANALYTICAL — yearly stats by variable")
print("="*60)

intangibles_stats = (
    df_intangibles_it[id_cols + data_cols]
    .melt(id_vars=["year"], value_vars=data_cols, var_name="var", value_name="value")
    .assign(value=lambda d: pd.to_numeric(d["value"], errors="coerce"))
    .groupby(["var", "year"])["value"]
    .agg(
        n="count",
        missing=lambda s: s.isna().sum(),
        mean="mean",
        median="median",
        std="std",
        min="min",
        max="max",
    )
    .reset_index()
    .sort_values(["var", "year"])
)

for var, grp in intangibles_stats.groupby("var"):
    print(f"\n— {var} —")
    print(grp.drop(columns="var").to_string(index=False))